# Tech Challenge Fase 2 – Demonstração do Pipeline Completo

**FIAP PosTech – IA para Devs (IADT)**  
**Projeto 1 – Otimização de Regressão Logística via Algoritmo Genético + LLM**

Este notebook demonstra o pipeline end-to-end:
1. Carregamento e pré-processamento do dataset Breast Cancer
2. Treinamento dos 3 modelos baseline (Módulo 1)
3. Otimização de hiperparâmetros via Algoritmo Genético (Módulo 2)
4. Geração de laudo interpretativo via LLM local (Ollama qwen3:8b)

> ⚠️ **Aviso:** As saídas da LLM são de **apoio à decisão clínica** e **não substituem** o julgamento de profissional de saúde habilitado.

In [ ]:
# Setup: adiciona a raiz do projeto ao path para importações absolutas
import sys
from pathlib import Path

# Navega da pasta notebooks/ até a raiz do projeto
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Raiz do projeto: {ROOT}")

In [ ]:
# Carrega variáveis de ambiente do .env (se existir)
from dotenv import load_dotenv
import os

load_dotenv(ROOT / ".env")

# Garante que o modo mock está ativo para não precisar do Ollama no notebook
# Mude para 'false' se quiser usar o Ollama real
os.environ.setdefault("LLM_MOCK", "true")

print(f"LLM_MOCK: {os.getenv('LLM_MOCK')}")
print(f"OLLAMA_MODEL: {os.getenv('OLLAMA_MODEL', 'qwen3:8b (padrão)')}")

---
## 1. Carregamento e Pré-processamento do Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from src.models.dataset import load_raw, prepare_data, NUMERIC_COLS, CATEGORICAL_COLS

# Carrega o CSV bruto para exploração
df = load_raw()
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Distribuição do target
print("Distribuição do target (Status):")
print(df["Status"].value_counts())
print(f"\nDesbalanceamento: {df['Status'].value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# Prepara os dados (split treino/teste estratificado)
X_train, X_test, y_train, y_test, le = prepare_data()

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"Classes: {list(le.classes_)} (0=Alive, 1=Dead)")
print(f"Dead no treino: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Dead no teste:  {y_test.sum()} ({y_test.mean():.1%})")

---
## 2. Módulo 1 – Modelos Baseline

Três classificadores treinados com configurações padrão + `class_weight="balanced"`.

In [ ]:
from src.models.train import run_baseline

# Treina os 3 modelos e salva métricas
# save=False para não sobrescrever os JSONs gerados pelos experimentos reais
baseline_results, trained_pipelines = run_baseline(save=False)

# Exibe comparativo
df_baseline = pd.DataFrame(baseline_results)
df_baseline = df_baseline.set_index("model")
print("\n=== Resultados Baseline ===")
df_baseline[["f1_dead", "roc_auc", "recall_dead", "accuracy", "cv_f1_mean"]]

In [ ]:
# Visualização das métricas baseline
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics_to_plot = ["f1_dead", "roc_auc", "recall_dead"]
titles = ["F1 (Dead)", "ROC-AUC", "Recall (Dead)"]
colors = ["#2196F3", "#4CAF50", "#FF5722"]

for ax, metric, title, color in zip(axes, metrics_to_plot, titles, colors):
    values = df_baseline[metric]
    bars = ax.bar(range(len(values)), values, color=color, alpha=0.8)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(values.index, rotation=15, ha='right', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 1.0)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.4f}", ha='center', va='bottom', fontsize=8)

fig.suptitle("Módulo 1 – Comparativo Baseline", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nConclusão: Regressão Logística selecionada para otimização (melhor F1 Dead + ROC-AUC)")

---
## 3. Módulo 2 – Otimização via Algoritmo Genético

O AG busca os melhores hiperparâmetros da Regressão Logística usando F1 Dead (5-fold StratCV) como fitness.

In [ ]:
from src.ga.engine import GAConfig, run_ga

# Configuração compacta para demo (populacao e gerações reduzidas para agilidade)
# Para resultados completos, veja experiments/run_experiments.py
config_demo = GAConfig(
    experiment_name="demo_notebook",
    pop_size=10,
    n_generations=15,
    mutation_rate=0.10,
    crossover_rate=0.80,
    tournament_size=3,
    elite_size=2,
    seed=42,
)

print("Executando AG (configuração reduzida para demo)...")
print(f"  pop_size={config_demo.pop_size} | gerações={config_demo.n_generations}")
print(f"  mutation_rate={config_demo.mutation_rate} | crossover_rate={config_demo.crossover_rate}")
print()

In [ ]:
# Roda o AG (pode levar 2-5 minutos dependendo da máquina)
best_individual, history = run_ga(config_demo, X_train, y_train)

print(f"\nMelhor indivíduo encontrado:")
print(f"  Genes (hiperparâmetros): {best_individual.genes}")
print(f"  Fitness (CV F1 Dead):    {best_individual.fitness:.4f}")

In [ ]:
# Curva de convergência
gens = [h.generation for h in history]
best_fit = [h.best_fitness for h in history]
mean_fit = [h.mean_fitness for h in history]

plt.figure(figsize=(9, 4))
plt.plot(gens, best_fit, 'b-o', markersize=4, label='Melhor fitness')
plt.plot(gens, mean_fit, 'g--s', markersize=4, label='Fitness médio')
plt.xlabel('Geração')
plt.ylabel('F1 Dead (5-fold CV)')
plt.title('Curva de Convergência do AG (demo)', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Avalia o melhor indivíduo no conjunto de teste
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, roc_auc_score, recall_score, accuracy_score
from src.models.dataset import build_preprocessor

genes = best_individual.genes
pipeline_otimizado = Pipeline([
    ("pre", build_preprocessor()),
    ("clf", LogisticRegression(
        C=genes["C"],
        penalty=genes["penalty"],
        solver=genes["solver"],
        max_iter=genes["max_iter"],
        tol=genes["tol"],
        class_weight="balanced",
        random_state=42,
    )),
])

pipeline_otimizado.fit(X_train, y_train)
y_pred_opt = pipeline_otimizado.predict(X_test)
y_proba_opt = pipeline_otimizado.predict_proba(X_test)[:, 1]

print("Resultados do modelo otimizado pelo AG no conjunto de teste:")
print(f"  F1 Dead:     {f1_score(y_test, y_pred_opt):.4f}")
print(f"  ROC-AUC:     {roc_auc_score(y_test, y_proba_opt):.4f}")
print(f"  Recall Dead: {recall_score(y_test, y_pred_opt):.4f}")
print(f"  Accuracy:    {accuracy_score(y_test, y_pred_opt):.4f}")
print(f"  Hiperparâmetros: {genes}")

In [ ]:
# Gráfico comparativo: baseline LR vs. AG otimizado (experimentos completos)
comparativo_img = ROOT / "experiments" / "comparativo_final.png"

if comparativo_img.exists():
    img = mpimg.imread(comparativo_img)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Comparativo Final – Baseline vs. Experimentos AG", fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Gráfico comparativo_final.png não encontrado.")
    print("Execute experiments/run_experiments.py para gerá-lo.")

---
## 4. Módulo 3 – Laudo Interpretativo via LLM

Usa o Ollama (qwen3:8b) para gerar uma explicação em linguagem natural do diagnóstico.  
Em modo mock (`LLM_MOCK=true`), retorna uma resposta simulada sem precisar do Ollama instalado.

> ⚠️ **Aviso:** As respostas geradas são de **apoio à decisão clínica** e **não substituem** o profissional de saúde.

In [ ]:
from src.llm.client import generate_explanation

# Seleciona uma amostra do conjunto de teste para demonstração
idx = 0  # altere para explorar outros pacientes
sample = X_test.iloc[idx].to_dict()
real_label = le.inverse_transform([y_test[idx]])[0]
pred_label = le.inverse_transform([y_pred_opt[idx]])[0]
prob_dead = float(y_proba_opt[idx])

print(f"Paciente selecionado (índice {idx}):")
print(f"  Rótulo real:   {real_label}")
print(f"  Predição:      {pred_label}")
print(f"  P(Dead):       {prob_dead:.1%}")
print()

# Métricas do modelo otimizado para incluir no laudo
model_metrics = {
    "f1_dead":     round(f1_score(y_test, y_pred_opt), 4),
    "roc_auc":     round(roc_auc_score(y_test, y_proba_opt), 4),
    "recall_dead": round(recall_score(y_test, y_pred_opt), 4),
}

In [ ]:
# Gera o laudo
# mock=True (padrão via env var LLM_MOCK) — não requer Ollama
# Para usar o Ollama real: defina LLM_MOCK=false no .env e rode `ollama serve`
mock_mode = os.getenv("LLM_MOCK", "true").lower() == "true"

print(f"Modo mock: {mock_mode}")
print("Gerando laudo...\n")
print("=" * 70)

laudo = generate_explanation(
    dados_paciente=sample,
    predicao=pred_label,
    probabilidade_dead=prob_dead,
    model_metrics=model_metrics,
    mock=mock_mode,
)

print(laudo)
print("=" * 70)

---
## Resumo Final

| Etapa | Status |
|-------|--------|
| Dataset carregado (4024 amostras, 16 colunas) | ✅ |
| Baseline: 3 modelos treinados e comparados | ✅ |
| AG: hiperparâmetros otimizados via evolução | ✅ |
| LLM: laudo interpretativo gerado | ✅ |

Para resultados completos dos 3 experimentos do AG, execute:
```bash
python experiments/run_experiments.py
```

Para usar a LLM real (sem mock):
```bash
ollama pull qwen3:8b
ollama serve
# Defina LLM_MOCK=false no .env
```